# 05 - Structured Extraction: Enrich Master Dataset via LLM+RAG

This notebook enriches `master_ai_trials_dataset.csv` by extracting structured fields from retrieved chunks for each NCT ID.

Fields extracted: primary/secondary outcomes, trial status, success prob, enrollment, dates, adverse events, drug names.

Steps:
1. Load master dataset and select NCTs.
2. For each NCT: retrieve top-20 chunks, extract structured JSON via LLM.
3. Normalize values, verify claims against chunks.
4. Save per-NCT JSONs and enriched CSV.

Outputs: per-NCT JSONs under `data/processed/extractions/`, `data/processed/enriched_master_ai_trials_dataset.csv`.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
import logging
from pathlib import Path
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Paths
DATA_PROCESSED = Path('../data/processed')
MASTER_CSV = Path('../data/processed/master_ai_trials_dataset.csv')
EXTRACTIONS_DIR = DATA_PROCESSED / 'extractions'
ENRICHED_CSV = DATA_PROCESSED / 'enriched_master_ai_trials_dataset.csv'
VSTORE_DIR = Path('../data/processed/vectorstore/chroma_db')

EXTRACTIONS_DIR.mkdir(exist_ok=True)

# Imports
from biotech_rag.extraction.structured_extraction import enrich_trial_data, save_enriched_csv
from biotech_rag.indexing.embedders import Embedder
from biotech_rag.generation.llm_clients import get_openrouter_llm
try:
    from biotech_rag.config import settings
except ImportError:
    settings = None

In [ ]:
# Load master dataset
master_df = pd.read_csv(MASTER_CSV)
ncts = master_df['nct_id'].dropna().unique()
logger.info(f"Loaded {len(ncts)} NCT IDs from master dataset.")
print(f"Sample NCTs: {list(ncts[:5])}")

In [ ]:
# Enrichment loop
embedder = Embedder()
llm = get_openrouter_llm()
enriched_records = []

sample_ncts = ncts[:min(len(ncts), settings.max_ncts_per_run if settings else 20)]
for nct in tqdm(sample_ncts, desc="Enriching trials"):
    try:
        # Check if vectorstore has data for this NCT
        if not validate_vectorstore_for_nct(nct, VSTORE_DIR):
            logger.warning(f"No chunks found for {nct}; skipping.")
            enriched_records.append({"nct_id": nct, "error": "no_chunks_in_vectorstore"})
            continue
        
        enriched = enrich_trial_data(nct, VSTORE_DIR, embedder, top_k=20)
        enriched_records.append(enriched)
        
        # Save per-NCT JSON
        with open(EXTRACTIONS_DIR / f"{nct}.json", 'w') as f:
            json.dump(enriched, f, indent=2)
        
    except Exception as e:
        logger.error(f"Failed to enrich {nct}: {e}")
        enriched_records.append({"nct_id": nct, "error": str(e)})

# Save enriched CSV
save_enriched_csv(enriched_records, ENRICHED_CSV)
logger.info(f"Enrichment complete. Processed {len(enriched_records)} trials.")

In [ ]:
# Quick summary
df_enriched = pd.read_csv(ENRICHED_CSV)
print(f"Enriched CSV shape: {df_enriched.shape}")
print(f"Columns: {list(df_enriched.columns)}")
print(df_enriched.head(2))